In [2]:
import os

# ── auto-detect project root (works on ANY computer!) ─────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR      = os.path.join(PROJECT_ROOT, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

print(f" Project root : {PROJECT_ROOT}")
print(f" Raw data dir : {RAW_DIR}")
print(" Paths ready!")

 Project root : D:\airline_pipeline
 Raw data dir : D:\airline_pipeline\data\raw
 Paths ready!


In [3]:
import os
import urllib.request

# ── BTS direct download links (2018 – 2023, all 12 months each) ──────
BASE = "https://transtats.bts.gov/PREZIP/"

files = []
for year in range(2018, 2024):        # 2018, 2019, 2020, 2021, 2022, 2023
    for month in range(1, 13):        # 1 through 12
        fname = f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        files.append((fname, f"{BASE}{fname}"))

# ── download function with progress ──────────────────────────────────
def download_file(filename, url, save_dir):
    save_path = os.path.join(save_dir, filename)

    if os.path.exists(save_path):
        print(f"    already exists, skipping  →  {filename}")
        return

    print(f"    downloading  →  {filename}")
    try:
        urllib.request.urlretrieve(url, save_path)
        size_mb = os.path.getsize(save_path) / (1024 * 1024)
        print(f"    saved  ({size_mb:.1f} MB)  →  {filename}")
    except Exception as e:
        print(f"    failed  →  {filename}  |  error: {e}")

# ── run all downloads ─────────────────────────────────────────────────
print(f" Saving to: {RAW_DIR}")
print(f" Total files to download: {len(files)}\n")

for i, (fname, url) in enumerate(files, 1):
    print(f"[{i}/{len(files)}]", end=" ")
    download_file(fname, url, RAW_DIR)

print("\n All downloads complete!")


 Saving to: D:\airline_pipeline\data\raw
 Total files to download: 72

[1/72]     already exists, skipping  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_1.zip
[2/72]     already exists, skipping  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_2.zip
[3/72]     downloading  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_3.zip
    saved  (28.8 MB)  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_3.zip
[4/72]     downloading  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_4.zip
    saved  (29.4 MB)  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_4.zip
[5/72]     downloading  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_5.zip
    saved  (30.3 MB)  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_5.zip
[6/72]     downloading  →  On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_6.zip
    saved  (30.7 MB)  →  On_Time_Rep

In [4]:
import os
import zipfile
import urllib.request
from datetime import datetime

today         = datetime.today()
CURRENT_YEAR  = today.year
CURRENT_MONTH = today.month
BASE          = "https://transtats.bts.gov/PREZIP/"

# ── build file list 2024 → now ────────────────────────────────────────
files = []
for year in range(2024, CURRENT_YEAR + 1):
    for month in range(1, 13):
        if year == CURRENT_YEAR and month > CURRENT_MONTH:
            break
        fname = f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        files.append((fname, f"{BASE}{fname}"))

# ── download ──────────────────────────────────────────────────────────
print("=" * 60)
print("PHASE 1 — Downloading 2024 to now")
print("=" * 60)

for i, (fname, url) in enumerate(files, 1):
    save_path = os.path.join(RAW_DIR, fname)
    if os.path.exists(save_path):
        print(f"[{i}/{len(files)}]   already exists → {fname}")
        continue
    print(f"[{i}/{len(files)}]   downloading → {fname}")
    try:
        urllib.request.urlretrieve(url, save_path)
        size_mb = os.path.getsize(save_path) / (1024 * 1024)
        print(f"[{i}/{len(files)}]   saved ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"[{i}/{len(files)}]   failed → {e}")

# ── unzip ALL zip files in raw folder ────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 2 — Unzipping ALL files in data/raw")
print("=" * 60)

all_zips = [f for f in os.listdir(RAW_DIR) if f.endswith(".zip")]
print(f" Found {len(all_zips)} zip files to extract\n")

for i, zip_name in enumerate(sorted(all_zips), 1):
    zip_path = os.path.join(RAW_DIR, zip_name)
    csv_name = zip_name.replace(".zip", ".csv")
    csv_path = os.path.join(RAW_DIR, csv_name)

    # skip if already extracted
    if os.path.exists(csv_path):
        print(f"[{i}/{len(all_zips)}]   already extracted → {csv_name}")
        continue

    print(f"[{i}/{len(all_zips)}]  extracting → {zip_name}")
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(RAW_DIR)
        size_mb = os.path.getsize(csv_path) / (1024 * 1024)
        print(f"[{i}/{len(all_zips)}]   extracted ({size_mb:.1f} MB) → {csv_name}")
    except Exception as e:
        print(f"[{i}/{len(all_zips)}]   error → {e}")

# ── delete zip files to save space ───────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 3 — Cleaning up zip files")
print("=" * 60)

deleted = 0
for zip_name in all_zips:
    zip_path = os.path.join(RAW_DIR, zip_name)
    try:
        os.remove(zip_path)
        print(f"    deleted → {zip_name}")
        deleted += 1
    except Exception as e:
        print(f"    could not delete → {zip_name} | {e}")

print(f"\n Deleted {deleted} zip files")

# ── final summary ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

csv_files = [f for f in os.listdir(RAW_DIR) if f.endswith(".csv")]
total_size = sum(os.path.getsize(os.path.join(RAW_DIR, f)) for f in csv_files)
print(f" Total CSV files : {len(csv_files)}")
print(f" Total data size : {total_size / (1024**3):.2f} GB")
print("\n All done! Ready for Step 4!")

PHASE 1 — Downloading 2024 to now
[1/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
[1/27]   saved (26.3 MB)
[2/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip
[2/27]   saved (23.9 MB)
[3/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip
[3/27]   saved (28.0 MB)
[4/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip
[4/27]   saved (27.2 MB)
[5/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip
[5/27]   saved (29.7 MB)
[6/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip
[6/27]   saved (29.8 MB)
[7/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip
[7/27]   saved (32.8 MB)
[8/27]   downloading → On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip
[8/27]   saved (29.4 MB)
[9/27]   downloading →

In [5]:
import os

RAW_DIR = r"D:\airline_pipeline\data\raw"

csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

print(f"✅ Total CSV files found: {len(csv_files)}")
print(f"\nFirst 3 files:")
for f in csv_files[:3]:
    print(f"  {f}")
print(f"\nLast 3 files:")
for f in csv_files[-3:]:
    print(f"  {f}")

total_gb = sum(os.path.getsize(os.path.join(RAW_DIR, f)) for f in csv_files) / (1024**3)
print(f"\n Total size: {total_gb:.2f} GB")

✅ Total CSV files found: 95

First 3 files:
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_1.csv
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_10.csv
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_11.csv

Last 3 files:
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_7.csv
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_8.csv
  On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_9.csv

 Total size: 22.04 GB


In [6]:
# ── verify all libraries installed correctly ──────────────────────
import importlib

libraries = {
    "pyspark"     : "PySpark (Spark engine)",
    "dask"        : "Dask (parallel computing)",
    "pandas"      : "Pandas (data manipulation)",
    "pyarrow"     : "PyArrow (Parquet support)",
    "matplotlib"  : "Matplotlib (charts)",
    "seaborn"     : "Seaborn (charts)",
    "plotly"      : "Plotly (interactive charts)",
    "findspark"   : "FindSpark (Spark finder)",
}

print("=" * 50)
print("LIBRARY CHECK")
print("=" * 50)

all_good = True
for lib, name in libraries.items():
    try:
        mod = importlib.import_module(lib)
        version = getattr(mod, "__version__", "ok")
        print(f"    {name:35s} v{version}")
    except ImportError:
        print(f"    {name:35s} NOT INSTALLED")
        all_good = False

print("=" * 50)
if all_good:
    print(" All libraries ready! Let's build the pipeline!")
else:
    print("  Some libraries missing — paste output here!")

LIBRARY CHECK
    PySpark (Spark engine)              v3.5.1
    Dask (parallel computing)           v2025.1.0
    Pandas (data manipulation)          v2.3.3
    PyArrow (Parquet support)           v21.0.0
    Matplotlib (charts)                 v3.10.6
    Seaborn (charts)                    v0.13.2
    Plotly (interactive charts)         v6.3.0
    FindSpark (Spark finder)            v2.0.1
 All libraries ready! Let's build the pipeline!


In [8]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AirlinePipeline") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("=" * 50)
print(f" Spark version  : {spark.version}")
print(f" Spark is alive : {spark.sparkContext.appName}")
print("=" * 50)
print(" Spark is ready!")

spark.stop()

 Spark version  : 3.5.1
 Spark is alive : AirlinePipeline
 Spark is ready!


In [10]:
import os

# ── DELETE 2018 and 2019 (oldest data) ───────────────────────────────
# ── KEEPING 2020, 2021, 2022, 2023, 2024, 2025 (recent 6 years) ──────

years_to_delete = [2018, 2019]

csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

deleted_files = 0
deleted_size  = 0

print("  Deleting old files (2018 & 2019)...")
print("=" * 55)

for f in csv_files:
    for year in years_to_delete:
        if f"present)_{year}_" in f:
            fpath     = os.path.join(RAW_DIR, f)
            size_mb   = os.path.getsize(fpath) / (1024 * 1024)
            os.remove(fpath)
            deleted_size  += size_mb
            deleted_files += 1
            print(f"    deleted → {f}  ({size_mb:.1f} MB)")

print("=" * 55)
print(f"  Deleted {deleted_files} files  ({deleted_size/1024:.2f} GB removed)")

# ── show what remains ─────────────────────────────────────────────────
remaining = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])
remaining_size = sum(
    os.path.getsize(os.path.join(RAW_DIR, f)) for f in remaining
) / (1024**3)

print(f"\n Remaining files : {len(remaining)}")
print(f" Remaining size  : {remaining_size:.2f} GB")
print(f"\n Years kept:")
years_kept = sorted(set(
    f.split("present)_")[1].split("_")[0]
    for f in remaining if "present)" in f
))
for y in years_kept:
    count = sum(1 for f in remaining if f"present)_{y}_" in f)
    print(f"   {y}  →  {count} months")

print("\n Done! Recent data ready for Spark!")

  Deleting old files (2018 & 2019)...
  Deleted 0 files  (0.00 GB removed)

 Remaining files : 72
 Remaining size  : 16.11 GB

 Years kept:
   2020  →  12 months
   2021  →  12 months
   2022  →  12 months
   2023  →  12 months
   2024  →  12 months
   2025  →  12 months

 Done! Recent data ready for Spark!


In [11]:
import os

# ── DELETE 2020 ───────────────────────────────────────────────────────
years_to_delete = [2020]

csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

deleted_files = 0
deleted_size  = 0

print("  Deleting 2020 files...")
print("=" * 55)

for f in csv_files:
    for year in years_to_delete:
        if f"present)_{year}_" in f:
            fpath   = os.path.join(RAW_DIR, f)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            os.remove(fpath)
            deleted_size  += size_mb
            deleted_files += 1
            print(f"  🗑️  deleted → {f}  ({size_mb:.1f} MB)")

print("=" * 55)
print(f"  Deleted {deleted_files} files  ({deleted_size/1024:.2f} GB removed)")

# ── show what remains ─────────────────────────────────────────────────
remaining = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])
remaining_size = sum(
    os.path.getsize(os.path.join(RAW_DIR, f)) for f in remaining
) / (1024**3)

print(f"\n Remaining files : {len(remaining)}")
print(f" Remaining size  : {remaining_size:.2f} GB")
print(f"\n Years kept:")
years_kept = sorted(set(
    f.split("present)_")[1].split("_")[0]
    for f in remaining if "present)" in f
))
for y in years_kept:
    count = sum(1 for f in remaining if f"present)_{y}_" in f)
    print(f"   {y}  →  {count} months")

print("\n Done!")

  Deleting 2020 files...
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_1.csv  (261.0 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_10.csv  (151.4 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_11.csv  (156.6 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_12.csv  (159.8 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_2.csv  (247.1 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_3.csv  (271.3 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_4.csv  (125.6 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_5.csv  (76.6 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_6.csv  (95.8 MB)
  🗑️  deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_presen

In [12]:
import os

# ── DELETE 2021 ───────────────────────────────────────────────────────
years_to_delete = [2021]

csv_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])

deleted_files = 0
deleted_size  = 0

print("  Deleting 2021 files...")
print("=" * 55)

for f in csv_files:
    for year in years_to_delete:
        if f"present)_{year}_" in f:
            fpath   = os.path.join(RAW_DIR, f)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            os.remove(fpath)
            deleted_size  += size_mb
            deleted_files += 1
            print(f"    deleted → {f}  ({size_mb:.1f} MB)")

print("=" * 55)
print(f"  Deleted {deleted_files} files  ({deleted_size/1024:.2f} GB removed)")

# ── show what remains ─────────────────────────────────────────────────
remaining = sorted([f for f in os.listdir(RAW_DIR) if f.endswith(".csv")])
remaining_size = sum(
    os.path.getsize(os.path.join(RAW_DIR, f)) for f in remaining
) / (1024**3)

print(f"\n Remaining files : {len(remaining)}")
print(f" Remaining size  : {remaining_size:.2f} GB")
print(f"\n Years kept:")
years_kept = sorted(set(
    f.split("present)_")[1].split("_")[0]
    for f in remaining if "present)" in f
))
for y in years_kept:
    count = sum(1 for f in remaining if f"present)_{y}_" in f)
    print(f"   {y}  →  {count} months")

print("\n Done!")

  Deleting 2021 files...
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_1.csv  (155.1 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_10.csv  (243.2 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_11.csv  (236.0 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_12.csv  (238.0 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_2.csv  (142.0 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_3.csv  (190.6 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_4.csv  (193.5 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_5.csv  (213.1 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_6.csv  (235.5 MB)
    deleted → On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2021_7.csv  (25